In [1]:
import pandas as pd
import numpy as np

# Load the labelled dataset
df = pd.read_csv("../data/processed/kerala_flood_labeled.csv")

# Define monthly rainfall columns
months = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"
]

# Define prediction cutoffs
cutoffs = {
    "March": months[:3],
    "May": months[:5],
    "June": months[:6],
    "July": months[:7],
    "September": months[:9]
}

# Create feature datasets
feature_sets = {}

for cutoff, columns in cutoffs.items():
    feature_sets[cutoff] = df[columns].copy()

    print(
        f"{cutoff}: {len(columns)} features | "
        f"Shape: {feature_sets[cutoff].shape}"
    )

# Keep target separate
y = df["FLOOD_LABEL"]

print("\nTarget distribution:")
print(y.value_counts())

print("\nFeature engineering setup completed!")

March: 3 features | Shape: (117, 3)
May: 5 features | Shape: (117, 5)
June: 6 features | Shape: (117, 6)
July: 7 features | Shape: (117, 7)
September: 9 features | Shape: (117, 9)

Target distribution:
FLOOD_LABEL
0    111
1      6
Name: count, dtype: int64

Feature engineering setup completed!


#### Feature Cutoff Validation

This section verifies the feature sets created for the five prediction cutoffs.

Each feature set must contain only monthly rainfall observations available up to its respective prediction cutoff.

Future-month rainfall, annual aggregates, seasonal aggregates and target labels must not be included as model input features.

Additional checks are performed to verify dataset dimensions, observation consistency, feature validity and target separation.

In [2]:
# FEATURE ENGINEERING VALIDATION

from pathlib import Path

# Define the permitted number of months for each cutoff
cutoff_month_counts = {
    "March": 3,
    "May": 5,
    "June": 6,
    "July": 7,
    "September": 9
}

# Columns that must never enter the prediction feature sets
excluded_columns = [
    "YEAR",
    "SUBDIVISION",
    "ANNUAL",
    "JF",
    "MAM",
    "JJAS",
    "OND",
    "FLOOD_LABEL"
]

# 1. Validate the original labelled dataset
assert len(df) == 117
assert df["YEAR"].is_unique

assert df["FLOOD_LABEL"].sum() == 6
assert (df["FLOOD_LABEL"] == 0).sum() == 111

# 2. Validate cutoff definitions
assert list(cutoffs.keys()) == list(cutoff_month_counts.keys())

assert list(feature_sets.keys()) == list(cutoffs.keys())

# 3. Validate feature sets individually
validation_records = []

for cutoff, feature_df in feature_sets.items():

    allowed_month_count = cutoff_month_counts[cutoff]

    expected_features = months[:allowed_month_count]

    actual_features = feature_df.columns.tolist()

    # Verify exact feature selection and chronological order
    assert actual_features == expected_features, (
        f"Incorrect features for {cutoff} cutoff."
    )

    # Verify that future months are excluded
    future_months = months[allowed_month_count:]

    assert not set(actual_features).intersection(future_months), (
        f"Future-month leakage detected in {cutoff}."
    )

    # Verify that non-predictor columns are excluded
    assert not set(actual_features).intersection(excluded_columns), (
        f"Excluded columns detected in {cutoff}."
    )

    # Verify dataset dimensions
    assert feature_df.shape == (
        len(df),
        allowed_month_count
    )

    # Verify observation alignment
    assert feature_df.index.equals(df.index)

    # Verify data quality
    assert feature_df.notna().all().all()

    values = feature_df.to_numpy(dtype=float)

    assert np.isfinite(values).all()

    assert (values >= 0).all()

    # Record validation summary
    validation_records.append({

        "Cutoff": cutoff,

        "Latest Available Month": expected_features[-1],

        "Feature Count": len(actual_features),

        "Observation Count": len(feature_df),

        "Flood Years": int(y.sum()),

        "Years Not Listed": int((y == 0).sum()),

        "Future Features Excluded": True,

        "Target Separated": "FLOOD_LABEL" not in actual_features
    })

# 4. Verify target separation
assert y.name == "FLOOD_LABEL"

assert y.index.equals(df.index)

assert y.isin([0, 1]).all()

# 5. Display validation summary
feature_validation_df = pd.DataFrame(
    validation_records
)

display(feature_validation_df)

# 6. Save validation results
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

feature_validation_df.to_csv(
    output_dir / "feature_cutoff_validation.csv",
    index=False
)

print("\nAll feature engineering checks passed!")

print("Cutoffs validated:", len(feature_sets))

print("Feature counts:",
      [len(feature_sets[name].columns) for name in cutoffs])

print("Target variable:", y.name)

print("Feature cutoff validation saved successfully!")

,Cutoff,Latest Available Month,Feature Count,Observation Count,Flood Years,Years Not Listed,Future Features Excluded,Target Separated
0,March,MAR,3,117,6,111,True,True
1,May,MAY,5,117,6,111,True,True
2,June,JUN,6,117,6,111,True,True
3,July,JUL,7,117,6,111,True,True
4,September,SEP,9,117,6,111,True,True



All feature engineering checks passed!
Cutoffs validated: 5
Feature counts: [3, 5, 6, 7, 9]
Target variable: FLOOD_LABEL
Feature cutoff validation saved successfully!
